# LCM-UNet — Bootstrap / Environment Sanity Check

One-time (or repeatable) check that: Drive mounts, the project tree exists
under `DRIVE_ROOT`, the GPU is visible, and the infra-layer pytest suite is
green. Safe to re-run — tree creation is idempotent (asserted below).

This does **not** run any model/training code.


In [ ]:
import os
import subprocess

IN_COLAB = os.path.isdir("/content")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = "https://github.com/Shihabul-Shuvo/LCM-UNet.git"
    REPO_DIR = "/content/LCM-UNet"
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    os.chdir(REPO_DIR)

    %pip install -q -r requirements.txt
else:
    print("Not running in Colab — assuming repo root is already the cwd (local dev).")

print("cwd:", os.getcwd())


In [ ]:
# Create (or confirm) the project tree under DRIVE_ROOT. Idempotent: run this
# cell as many times as you like, it will never duplicate or clobber files.
from lcmunet.paths import get_paths, resolve_drive_root, SUBDIRS

paths_before = get_paths()
listing_before = sorted(str(p) for p in paths_before.root.rglob("*"))

paths_after = get_paths()  # second call, same session -> must be a no-op
listing_after = sorted(str(p) for p in paths_after.root.rglob("*"))

assert paths_before == paths_after
assert listing_before == listing_after, "tree creation is not idempotent!"

print("DRIVE_ROOT:", resolve_drive_root())
for name in SUBDIRS:
    d = getattr(paths_after, name)
    print(f"  {name:12s} -> {d}  (exists={d.is_dir()})")
print("Idempotency check: OK (running this cell twice created nothing new).")


In [ ]:
# GPU name + free VRAM (informational only here; real GPU gates are run and
# pasted by the user, per project rules — this notebook does not claim them).
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    free_b, total_b = torch.cuda.mem_get_info(0)
    print(f"GPU: {name}")
    print(f"Free VRAM:  {free_b / 1024**3:.2f} GB")
    print(f"Total VRAM: {total_b / 1024**3:.2f} GB")
else:
    print("No CUDA GPU visible in this runtime.")


In [ ]:
# Run the infra-layer pytest suite.
result = subprocess.run(["python", "-m", "pytest", "-q"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "pytest failed — see output above"
print("pytest: all green.")
